In [8]:
import numpy as np
import pandas as pd
from tqdm import tqdm 

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import h5py

plt.rcParams["font.sans-serif"] = "SimHei"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 300

import sys
sys.path.append("D:/CPResearch/utils")
from utils import backtest

import os

In [ ]:
from alphalab.backtest import run_backtest

In [3]:
def load_data(fct_name):
    h5_path = f"D:/CPResearch/market_data/XYQuantData/HDF5Data/ElementaryFactor/{fct_name}.hdf5"
    with h5py.File(h5_path, "r") as f:
        data = f["Data"][:]
        codes = f["ID"].asstr()[:]
        dates = pd.to_datetime(f["DateTime"][:], unit="s") + pd.Timedelta(hours=8)  # 将日期字符串转为标准日期格式, UTC+8
    df = pd.DataFrame(index=dates, columns=codes, data=data)
    df = df.sort_index()
    return df

df_stock_price = load_data("复权收盘价")
df_stock_price

,000001.SZ,000002.SZ,000003.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,...,603293.SH,920012.BJ,301513.SZ,301666.SZ,920177.BJ,688808.SH,920125.BJ,001312.SZ,920191.BJ,688820.SH
2000-01-04,396.190902,87.676742,22.004215,27.194720,27.664279,39.876613,30.075966,94.767241,10.245706,20.121468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,391.208731,85.463542,21.774206,28.065946,27.708613,39.506624,29.966995,96.328758,10.274245,20.417768,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,406.805092,89.464326,22.425899,29.248326,28.683956,40.698811,30.802438,98.713238,10.645260,20.956496,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,423.267918,93.550232,23.460940,30.710742,29.969635,42.959856,33.889947,103.292283,10.873576,21.818460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-10,436.264886,102.905248,24.380977,32.235389,31.077980,44.480923,34.907008,110.720043,11.016274,22.411060,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-20,913.678932,714.095883,NaN,14.507987,NaN,386.667038,114.561192,60.950557,80.416507,42.020741,...,60.75,62.90,96.56,237.99,38.35,NaN,NaN,NaN,14.88,NaN
2026-04-21,915.331154,708.644769,NaN,13.776492,NaN,384.282657,113.732839,60.502391,79.966247,40.296813,...,57.70,64.82,100.02,214.55,37.23,NaN,NaN,30.57,13.18,76.65
2026-04-22,907.070043,705.010694,NaN,13.085636,NaN,387.859228,114.644028,60.502391,80.776716,40.189068,...,57.80,64.83,94.71,224.90,37.53,NaN,NaN,29.10,13.19,95.00
2026-04-23,908.722265,692.291428,NaN,12.435418,NaN,386.269641,113.318663,59.830142,79.966247,39.865831,...,53.77,60.18,85.92,221.99,34.55,NaN,NaN,26.93,12.63,88.60


In [4]:
def load_index_weight(index_code):
    df_wgt = pd.read_parquet(f"D:/XYQuant2026/20260521_社保_沪深300指数分析修改V2/data/{index_code}.parquet")
    df_wgt = df_wgt.loc[df_wgt["数据来源"]=="指数提供商"]
    df_wgt["代码"] = df_wgt["代码"].str.slice(-6) + "." + df_wgt["代码"].str.slice(0, 2)
    df_wgt["权重"] = df_wgt["比例(%)"] / 100
    df_wgt["日期"] = pd.to_datetime(df_wgt["截止日"], format="%Y%m%d")
    return df_wgt

df_wgt300 = load_index_weight("SH000300")
df_wgt500 = load_index_weight("SH000905")
df_wgt800 = load_index_weight("SH000906")

In [6]:
port_ret_list = []
for index_code in ["SH000300", "SH000905", "SH000906"]:
    df_wgt = load_index_weight(index_code)
    s_port_ret = run_backtest(
        df_wgt, 
        df_stock_price.ffill(), 
        end_date="2026-04-30", 
        date_col="日期", code_col="代码", weight_col="权重"
    ).returns
    s_port_ret.name = index_code
    port_ret_list.append(s_port_ret)

df_port_ret = pd.concat(port_ret_list, axis=1)
df_port_ret = df_port_ret.rename(
    columns={"SH000300": "300收益", "SH000905": "500收益", "SH000906": "800收益"}
).loc[pd.to_datetime("2015-01-01"):]

D:\CPResearch\alphalab\alphalab\backtest\vectorized.py:59: UserWarning: Codes in target weights not found in price data: ['000022.SZ', '600849.SH']. These codes will be ignored.
  warnings.warn(f"Codes in target weights not found in price data: {sorted(missing_codes)}. These codes will be ignored.")
回测: 100%|██████████| 258/258 [00:01<00:00, 166.42it/s]
D:\CPResearch\alphalab\alphalab\backtest\vectorized.py:59: UserWarning: Codes in target weights not found in price data: ['000022.SZ', '600849.SH']. These codes will be ignored.
  warnings.warn(f"Codes in target weights not found in price data: {sorted(missing_codes)}. These codes will be ignored.")
回测: 100%|██████████| 237/237 [00:01<00:00, 137.40it/s]
D:\CPResearch\alphalab\alphalab\backtest\vectorized.py:59: UserWarning: Codes in target weights not found in price data: ['000022.SZ', '600849.SH']. These codes will be ignored.
  warnings.warn(f"Codes in target weights not found in price data: {sorted(missing_codes)}. These codes will b

In [10]:
backtest(df_port_ret).display()

,开始日期,结束日期,区间收益,年化收益率,年化波动率,夏普比率,最大回撤,卡玛比率
300收益,2015-01-05,2026-04-24,75.74%,5.11%,21.52%,0.24,45.92%,0.11
500收益,2015-01-05,2026-04-24,80.63%,5.37%,25.35%,0.21,63.95%,0.08
800收益,2015-01-05,2026-04-24,76.72%,5.17%,21.87%,0.24,48.37%,0.11


In [11]:
df_port_ret.std() * np.sqrt(250)

300收益    0.214295
500收益    0.252459
800收益    0.217853
dtype: float64